# LeetCode 2432 — The Employee That Worked on the Longest Task bằng Python

Link đề: https://leetcode.com/problems/the-employee-that-worked-on-the-longest-task/ · Độ khó: **Easy** · Chủ đề: Array

Skeleton LeetCode:

```python
class Solution:
    def hardestWorker(self, n: int, logs: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán: từ **nhật ký thời điểm kết thúc** từng công việc, tính **thời lượng** của từng công việc rồi tìm nhân viên làm công việc **dài nhất** (hòa thì lấy `id` nhỏ nhất).
2. Biết tính thời lượng bằng hiệu `leaveTime[i] - leaveTime[i-1]`, với công việc đầu tiên bắt đầu ở thời điểm `0`.
3. Nắm kỹ thuật **một lượt duyệt** có biến `prev` (thời điểm kết thúc trước) và quy tắc phá hòa theo `id`.
4. Viết 5 cách: tính danh sách thời lượng rồi tìm hai lượt, một lượt, sắp xếp theo `(-thời lượng, id)`, dùng `max` với khóa `(thời lượng, -id)`, và dùng dict theo `id`; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: **cộng dồn** thời gian theo từng nhân viên (đề hỏi từng công việc), quên điểm bắt đầu `0`, chọn `id` lớn khi hòa, và nhầm thời điểm kết thúc với thời lượng.

## 2. Đề bài & ví dụ

Có `n` nhân viên, mỗi người có `id` duy nhất từ `0` đến `n - 1`. Cho `logs`, trong đó `logs[i] = [id_i, leaveTime_i]`:

- `id_i` là `id` của nhân viên làm **công việc thứ `i`**.
- `leaveTime_i` là **thời điểm** nhân viên đó **hoàn thành** công việc thứ `i`.

Tất cả `leaveTime_i` **khác nhau** và **tăng dần**. Công việc đầu tiên bắt đầu ở thời điểm `0`; mỗi công việc sau bắt đầu ngay khi công việc trước kết thúc. Trả về `id` của nhân viên làm **một công việc có thời lượng dài nhất**. Nếu nhiều nhân viên cùng thời lượng dài nhất, trả về `id` **nhỏ nhất**.

| Ví dụ | n | logs | Output | Giải thích |
|---|---|---|---|---|
| 1 | `10` | `[[0,3],[2,5],[0,9],[1,15]]` | `1` | thời lượng `3, 2, 4, 6` (id `0,2,0,1`); dài nhất `6` của id `1` |
| 2 | `26` | `[[1,1],[3,7],[2,12],[7,17]]` | `3` | thời lượng `1, 6, 5, 5`; dài nhất `6` của id `3` |
| 3 | `2` | `[[0,10],[1,20]]` | `0` | thời lượng `10, 10` hòa; chọn id nhỏ nhất `0` |

Constraints: `2 <= n <= 500`; `1 <= logs.length <= 500`; `logs[i].length == 2`; `0 <= id_i <= n - 1`; `1 <= leaveTime_i <= 500`; `id_i != id_{i+1}`; `leaveTime_i` tăng nghiêm ngặt.

In [1]:
n, logs = 10, [[0, 3], [2, 5], [0, 9], [1, 15]]
prev = 0
for i, (emp, leave) in enumerate(logs):
    print(f"công việc {i}: id={emp}, bắt đầu={prev}, kết thúc={leave}, thời lượng={leave - prev}")
    prev = leave

công việc 0: id=0, bắt đầu=0, kết thúc=3, thời lượng=3
công việc 1: id=2, bắt đầu=3, kết thúc=5, thời lượng=2
công việc 2: id=0, bắt đầu=5, kết thúc=9, thời lượng=4
công việc 3: id=1, bắt đầu=9, kết thúc=15, thời lượng=6


## 3. Kiến thức nền

### 3.1. Danh sách các cặp và gán bộ (tuple unpacking)

`logs` là danh sách các danh sách con `[id, leave]`. Với `for emp, leave in logs:` Python tự **tách** từng cặp vào hai biến. `enumerate` cho thêm chỉ số.

In [2]:
logs = [[0, 3], [2, 5], [0, 9], [1, 15]]
for emp, leave in logs:
    print("id", emp, "kết thúc lúc", leave)
for i, (emp, leave) in enumerate(logs):
    print(i, emp, leave)

id 0 kết thúc lúc 3
id 2 kết thúc lúc 5
id 0 kết thúc lúc 9
id 1 kết thúc lúc 15
0 0 3
1 2 5
2 0 9
3 1 15


### 3.2. Thời lượng là hiệu hai thời điểm liên tiếp

Công việc `i` kéo dài từ `leaveTime_{i-1}` đến `leaveTime_i`; công việc `0` bắt đầu ở `0`. Do đó thời lượng `d_i = leaveTime_i - leaveTime_{i-1}` với quy ước `leaveTime_{-1} = 0`. Để tránh xử lý riêng công việc đầu, ta dùng biến `prev = 0`.

In [3]:
leaves = [3, 5, 9, 15]
prev = 0
durations = []
for t in leaves:
    durations.append(t - prev)
    prev = t
print(durations)
print([b - a for a, b in zip([0] + leaves, leaves)])     # cùng kết quả bằng zip

[3, 2, 4, 6]
[3, 2, 4, 6]


### 3.3. `max` với `key` và so sánh tuple

`max(iterable, key=f)` chọn phần tử có `f(x)` lớn nhất. Khi `f` trả về tuple, Python so sánh **từng thành phần từ trái sang phải**. Dùng khóa `(thời lượng, -id)`: thời lượng lớn hơn thắng; hòa thì `-id` lớn hơn thắng, tức `id` **nhỏ** hơn thắng.

In [4]:
pairs = [(0, 10), (1, 10)]            # (id, thời lượng)
print(max(pairs, key=lambda p: (p[1], -p[0])))      # (0, 10): hòa, id nhỏ hơn
print(max(pairs, key=lambda p: p[1]))               # chỉ so thời lượng -> (0, 10) vì max giữ phần tử đầu khi hòa

(0, 10)
(0, 10)


### 3.4. `sorted` với khóa kép

`sorted(items, key=lambda p: (-p[1], p[0]))` sắp xếp thời lượng **giảm dần**, hòa thì `id` **tăng dần**. Phần tử đầu danh sách là đáp án.

In [5]:
items = [(1, 6), (0, 6), (2, 4)]       # (id, thời lượng)
items.sort(key=lambda p: (-p[1], p[0]))
print(items)
print(items[0][0])

[(0, 6), (1, 6), (2, 4)]
0


### 3.5. Dict theo `id` và `setdefault` / `get`

`d.get(k, 0)` trả `0` nếu thiếu khóa. Với "một nhân viên làm nhiều công việc", ta có thể giữ cho mỗi `id` **thời lượng lớn nhất** của một công việc riêng lẻ (không cộng dồn).

In [6]:
best = {}
for emp, dur in [(0, 3), (2, 2), (0, 4), (1, 6)]:
    best[emp] = max(best.get(emp, 0), dur)
print(best)

{0: 4, 2: 2, 1: 6}


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Vẽ trục thời gian cho ví dụ 1: `logs = [[0,3],[2,5],[0,9],[1,15]]`.

```
thời gian: 0---3---5-------9-----------15
nhân viên:   0   2    0          1
thời lượng:  3   2    4          6
```

- Việc 0: id `0`, từ `0` đến `3` ⇒ `3`.
- Việc 1: id `2`, từ `3` đến `5` ⇒ `2`.
- Việc 2: id `0`, từ `5` đến `9` ⇒ `4`.
- Việc 3: id `1`, từ `9` đến `15` ⇒ `6`.

Lớn nhất là `6` của id `1`.

Lưu ý quan trọng: id `0` làm hai việc (`3` và `4`). Đề hỏi **công việc dài nhất**, không phải **tổng thời gian** của một nhân viên. Nếu cộng dồn, id `0` được `7 > 6`, sẽ cho đáp án sai.

In [7]:
logs = [[0, 3], [2, 5], [0, 9], [1, 15]]
prev, best_id, best_dur = 0, -1, -1
for emp, leave in logs:
    dur = leave - prev
    print(f"id={emp}: thời lượng {dur}")
    if dur > best_dur or (dur == best_dur and emp < best_id):
        best_id, best_dur = emp, dur
    prev = leave
print("đáp án:", best_id, "với thời lượng", best_dur)

id=0: thời lượng 3
id=2: thời lượng 2
id=0: thời lượng 4
id=1: thời lượng 6
đáp án: 1 với thời lượng 6


## 5. Cách 1 — Brute Force / baseline: tính danh sách thời lượng rồi tìm hai lượt

**Ý tưởng:** (1) Tạo danh sách `durations`. (2) Tìm giá trị lớn nhất `m = max(durations)`. (3) Duyệt các công việc, lấy **`id` nhỏ nhất** trong các công việc có thời lượng `m`.

In [8]:
from typing import List

class SolutionBrute:
    def hardestWorker(self, n: int, logs: List[List[int]]) -> int:
        durations = []
        prev = 0
        for emp, leave in logs:
            durations.append(leave - prev)
            prev = leave
        longest = max(durations)
        candidates = [logs[i][0] for i in range(len(logs)) if durations[i] == longest]
        return min(candidates)

In [9]:
sol = SolutionBrute()
print(sol.hardestWorker(10, [[0, 3], [2, 5], [0, 9], [1, 15]]))     # 1
print(sol.hardestWorker(26, [[1, 1], [3, 7], [2, 12], [7, 17]]))    # 3
print(sol.hardestWorker(2, [[0, 10], [1, 20]]))                     # 0

1
3
0


**Độ phức tạp:** Time `O(m)` với `m = len(logs)` (ba lượt tuyến tính), Space `O(m)` cho `durations`. **Nút thắt:** dùng bộ nhớ phụ và nhiều lượt; có thể gộp thành một lượt với hai biến.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** không cần lưu mọi thời lượng, chỉ cần nhớ **đáp án tốt nhất hiện tại**.

- **Invariant:** sau khi xử lý `i` công việc đầu, `(best_id, best_dur)` là cặp tốt nhất trong chúng: thời lượng lớn nhất, hòa thì `id` nhỏ nhất; `prev` là thời điểm kết thúc của công việc gần nhất.
- **Quy tắc cập nhật** với công việc mới `(emp, leave)`, `dur = leave - prev`:
  - **nếu** `dur > best_dur` **thì** lấy `(emp, dur)`;
  - **nếu** `dur == best_dur` **và** `emp < best_id` **thì** cũng lấy `(emp, dur)`;
  - rồi `prev = leave`.
- Khởi tạo `prev = 0`, `best_dur = -1` (hoặc `0` vì thời lượng thật `>= 1`), `best_id = n` hoặc `-1` và để điều kiện đầu tiên luôn thắng.

## 7. Các cách tiếp theo

### Cách 2 — Một lượt với biến `prev`

In [10]:
from typing import List

class SolutionOnePass:
    def hardestWorker(self, n: int, logs: List[List[int]]) -> int:
        prev = 0
        best_id, best_dur = -1, 0
        for emp, leave in logs:
            dur = leave - prev
            if dur > best_dur or (dur == best_dur and emp < best_id):
                best_id, best_dur = emp, dur
            prev = leave
        return best_id

In [11]:
print(SolutionOnePass().hardestWorker(10, [[0, 3], [2, 5], [0, 9], [1, 15]]))     # 1
print(SolutionOnePass().hardestWorker(26, [[1, 1], [3, 7], [2, 12], [7, 17]]))    # 3
print(SolutionOnePass().hardestWorker(2, [[0, 10], [1, 20]]))                     # 0

1
3
0


**Độ phức tạp:** Time `O(m)`, Space `O(1)`.

### Cách 3 — Sắp xếp theo `(-thời lượng, id)`

Tạo danh sách `(id, thời lượng)`, sắp xếp giảm theo thời lượng và tăng theo `id`; phần tử đầu tiên là đáp án.

In [12]:
from typing import List

class SolutionSort:
    def hardestWorker(self, n: int, logs: List[List[int]]) -> int:
        prevs = [0] + [t for _, t in logs[:-1]]
        tasks = [(emp, leave - p) for (emp, leave), p in zip(logs, prevs)]
        tasks.sort(key=lambda x: (-x[1], x[0]))
        return tasks[0][0]

In [13]:
print(SolutionSort().hardestWorker(10, [[0, 3], [2, 5], [0, 9], [1, 15]]))     # 1
print(SolutionSort().hardestWorker(26, [[1, 1], [3, 7], [2, 12], [7, 17]]))    # 3
print(SolutionSort().hardestWorker(2, [[0, 10], [1, 20]]))                     # 0

1
3
0


**Độ phức tạp:** Time `O(m log m)`, Space `O(m)`.

### Cách 4 — `max` với khóa `(thời lượng, -id)`

Một dòng: tìm cặp tốt nhất rồi lấy `id`. Dùng `zip` với `prev` để tính thời lượng mà không cần chỉ số.

In [14]:
from typing import List

class SolutionMax:
    def hardestWorker(self, n: int, logs: List[List[int]]) -> int:
        prevs = [0] + [t for _, t in logs]
        best = max(zip(logs, prevs), key=lambda p: (p[0][1] - p[1], -p[0][0]))
        return best[0][0]

In [15]:
print(SolutionMax().hardestWorker(10, [[0, 3], [2, 5], [0, 9], [1, 15]]))     # 1
print(SolutionMax().hardestWorker(26, [[1, 1], [3, 7], [2, 12], [7, 17]]))    # 3
print(SolutionMax().hardestWorker(2, [[0, 10], [1, 20]]))                     # 0

1
3
0


**Độ phức tạp:** Time `O(m)`, Space `O(m)` (danh sách `prevs`). Ngắn nhưng khó đọc hơn cách một lượt.

### Cách 5 — Dict: thời lượng một-việc dài nhất của mỗi nhân viên

Gom theo `id`: `best[emp] = max thời lượng của một việc` rồi chọn nhân viên tốt nhất. Cách này hợp khi bạn muốn **báo cáo theo nhân viên** (cần thêm thông tin từng người).

In [16]:
from typing import List

class SolutionDict:
    def hardestWorker(self, n: int, logs: List[List[int]]) -> int:
        best = {}
        prev = 0
        for emp, leave in logs:
            best[emp] = max(best.get(emp, 0), leave - prev)
            prev = leave
        top = max(best.values())
        return min(emp for emp, d in best.items() if d == top)

In [17]:
print(SolutionDict().hardestWorker(10, [[0, 3], [2, 5], [0, 9], [1, 15]]))     # 1
print(SolutionDict().hardestWorker(26, [[1, 1], [3, 7], [2, 12], [7, 17]]))    # 3
print(SolutionDict().hardestWorker(2, [[0, 10], [1, 20]]))                     # 0

1
3
0


**Độ phức tạp:** Time `O(m)`, Space `O(k)` với `k` là số nhân viên phân biệt (`k <= min(n, m)`).

## 8. TRACE từng bước

Cách một lượt, `logs = [[0,3],[2,5],[0,9],[1,15]]`:

| Việc | `emp` | `leave` | `prev` trước | `dur` | `best_id` trước | `best_dur` trước | Cập nhật? |
|---|---|---|---|---|---|---|---|
| 0 | 0 | 3 | 0 | 3 | -1 | 0 | `3 > 0` ⇒ `(0, 3)` |
| 1 | 2 | 5 | 3 | 2 | 0 | 3 | không |
| 2 | 0 | 9 | 5 | 4 | 0 | 3 | `4 > 3` ⇒ `(0, 4)` |
| 3 | 1 | 15 | 9 | 6 | 0 | 4 | `6 > 4` ⇒ `(1, 6)` |

Kết quả `1`.

In [18]:
def trace_hardest(n, logs):
    prev = 0
    best_id, best_dur = -1, 0
    for i, (emp, leave) in enumerate(logs):
        dur = leave - prev
        print(f"việc {i}: emp={emp}, leave={leave}, prev={prev}, dur={dur}, trước: best=({best_id}, {best_dur})", end=" -> ")
        if dur > best_dur or (dur == best_dur and emp < best_id):
            best_id, best_dur = emp, dur
            print("cập nhật thành", (best_id, best_dur))
        else:
            print("giữ nguyên")
        prev = leave
    return best_id

print("đáp án:", trace_hardest(10, [[0, 3], [2, 5], [0, 9], [1, 15]]))
print("đáp án:", trace_hardest(2, [[0, 10], [1, 20]]))

việc 0: emp=0, leave=3, prev=0, dur=3, trước: best=(-1, 0) -> cập nhật thành (0, 3)
việc 1: emp=2, leave=5, prev=3, dur=2, trước: best=(0, 3) -> giữ nguyên
việc 2: emp=0, leave=9, prev=5, dur=4, trước: best=(0, 3) -> cập nhật thành (0, 4)
việc 3: emp=1, leave=15, prev=9, dur=6, trước: best=(0, 4) -> cập nhật thành (1, 6)
đáp án: 1
việc 0: emp=0, leave=10, prev=0, dur=10, trước: best=(-1, 0) -> cập nhật thành (0, 10)
việc 1: emp=1, leave=20, prev=10, dur=10, trước: best=(0, 10) -> giữ nguyên
đáp án: 0


## 9. Code nộp LeetCode

Dùng cách một lượt:

In [19]:
from typing import List

class Solution:
    def hardestWorker(self, n: int, logs: List[List[int]]) -> int:
        prev = 0
        best_id, best_dur = -1, 0
        for emp, leave in logs:
            dur = leave - prev
            if dur > best_dur or (dur == best_dur and emp < best_id):
                best_id, best_dur = emp, dur
            prev = leave
        return best_id

In [20]:
print(Solution().hardestWorker(10, [[0, 3], [2, 5], [0, 9], [1, 15]]))
print(Solution().hardestWorker(26, [[1, 1], [3, 7], [2, 12], [7, 17]]))
print(Solution().hardestWorker(2, [[0, 10], [1, 20]]))

1
3
0


## 10. Vì sao thuật toán đúng

1. **Thời lượng đúng.** Vì công việc `i` bắt đầu ngay khi công việc `i-1` kết thúc (hoặc ở `0` cho việc đầu), thời lượng của nó là `leave_i - leave_{i-1}` với `leave_{-1} = 0`. Biến `prev` luôn bằng `leave_{i-1}` khi xử lý việc `i`, nên `dur` đúng.
2. **Invariant.** Sau khi xử lý việc `0..i`, `(best_id, best_dur)` là cặp `(id, thời lượng)` lớn nhất theo thứ tự `(thời lượng lớn hơn, rồi id nhỏ hơn)` trong số các việc đó. Ban đầu chưa có việc nào, `best_dur = 0` nhỏ hơn mọi thời lượng thật (`>= 1` vì thời điểm tăng nghiêm ngặt) nên việc đầu tiên luôn thay thế.
3. **Bước cập nhật.** Việc mới tốt hơn việc tốt nhất cũ khi thời lượng lớn hơn, hoặc bằng nhau mà `id` nhỏ hơn. Quy tắc if-or trên đúng bằng định nghĩa này nên invariant được giữ.
4. **Kết thúc.** Sau việc cuối, `best_id` là đáp án.
5. **Không cộng dồn.** Đề hỏi *công việc* dài nhất, không phải *nhân viên* làm tổng dài nhất. Mỗi việc được so sánh độc lập; cùng một `id` có thể xuất hiện nhiều lần và mỗi lần được xét riêng.
6. Các cách khác (hai lượt, sắp xếp, `max`, dict) thực hiện cùng phép chọn theo thứ tự ưu tiên đó.

## 11. Phân tích Time / Space Complexity

Đặt `m = len(logs)` và `k` là số nhân viên xuất hiện.

| Cách | Time | Space |
|---|---|---|
| Hai lượt với danh sách thời lượng | `O(m)` | `O(m)` |
| Một lượt với `prev` | `O(m)` | `O(1)` |
| Sắp xếp `(-dur, id)` | `O(m log m)` | `O(m)` |
| `max` với khóa | `O(m)` | `O(m)` |
| Dict theo `id` | `O(m)` | `O(k)` |

Với `m <= 500` cách nào cũng tức thì. Cách một lượt tốt nhất về bộ nhớ và dễ chứng minh.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

In [21]:
sols = [SolutionBrute(), SolutionOnePass(), SolutionSort(), SolutionMax(), SolutionDict(), Solution()]
cases = [
    ((10, [[0, 3], [2, 5], [0, 9], [1, 15]]), 1),          # ví dụ 1
    ((26, [[1, 1], [3, 7], [2, 12], [7, 17]]), 3),         # ví dụ 2
    ((2, [[0, 10], [1, 20]]), 0),                          # ví dụ 3: hòa -> id nhỏ
    ((2, [[1, 5]]), 1),                                    # một việc
    ((2, [[0, 1]]), 0),                                    # một việc, thời lượng 1
    ((5, [[3, 4], [1, 8]]), 1),                            # hòa 4 và 4: id nhỏ nhất là 1
    ((5, [[1, 4], [3, 8]]), 1),                            # hòa nhưng id nhỏ xuất hiện trước
    ((5, [[4, 2], [0, 3], [4, 10]]), 4),                   # việc 3 dài nhất thuộc id 4
    ((3, [[0, 1], [1, 2], [0, 3], [2, 4]]), 0),            # tất cả thời lượng 1 -> id nhỏ nhất 0
    ((500, [[499, 500]]), 499),                            # biên lớn nhất
    ((10, [[0, 3], [2, 5], [0, 9], [1, 15], [2, 21]]), 1),  # hòa 6 giữa id 1 và id 2 -> 1
    ((3, [[2, 7], [1, 8], [0, 9]]), 2),                    # việc đầu dài nhất
]
for (n, logs), want in cases:
    for s in sols:
        assert s.hardestWorker(n, logs) == want, (type(s).__name__, n, logs, s.hardestWorker(n, logs), want)
print("tất cả test cố định đều đúng với cả 6 cách:", len(cases))

tất cả test cố định đều đúng với cả 6 cách: 12


### 12.2. Đối chiếu ngẫu nhiên (có seed)

Sinh dữ liệu đúng constraints: `2 <= n <= 500`, `1 <= m <= 500`, `leaveTime` tăng nghiêm ngặt trong `[1, 500]` (chọn ngẫu nhiên tập phân biệt rồi sắp xếp), `id` hai việc liên tiếp khác nhau.

In [22]:
import random
random.seed(2432)

def gen_case(max_n, max_m):
    n = random.randint(2, max_n)
    m = random.randint(1, max_m)
    leaves = sorted(random.sample(range(1, 501), m))
    ids = []
    for _ in range(m):
        while True:
            x = random.randint(0, n - 1)
            if not ids or x != ids[-1]:
                ids.append(x)
                break
    return n, [[i, t] for i, t in zip(ids, leaves)]

def check_constraints(n, logs):
    assert 2 <= n <= 500 and 1 <= len(logs) <= 500
    for k, (i, t) in enumerate(logs):
        assert 0 <= i <= n - 1 and 1 <= t <= 500
        if k:
            assert i != logs[k - 1][0] and t > logs[k - 1][1]

for _ in range(1000):
    n, logs = gen_case(random.choice([2, 3, 5, 20, 500]), random.choice([1, 2, 5, 20, 500]))
    check_constraints(n, logs)
    results = {type(s).__name__: s.hardestWorker(n, logs) for s in sols}
    assert len(set(results.values())) == 1, (n, logs, results)
print("1000 test ngẫu nhiên: 6 cách khớp nhau")

1000 test ngẫu nhiên: 6 cách khớp nhau


### 12.3. Phản ví dụ: cộng dồn theo nhân viên cho đáp án khác

Với ví dụ 1, tổng thời gian của id `0` là `7` lớn hơn việc dài nhất `6`. Kiểm tra rằng đáp án đúng không phải id `0`, và kiểm tra tính chất: thời lượng của việc mà ta trả về là lớn nhất.

In [23]:
n, logs = 10, [[0, 3], [2, 5], [0, 9], [1, 15]]
assert Solution().hardestWorker(n, logs) == 1

random.seed(77)
for _ in range(300):
    n, logs = gen_case(20, 30)
    ans = Solution().hardestWorker(n, logs)
    prev, durs = 0, []
    for emp, t in logs:
        durs.append((emp, t - prev))
        prev = t
    top = max(d for _, d in durs)
    assert (ans, top) in durs                                  # ans có một việc đạt thời lượng lớn nhất
    assert ans == min(e for e, d in durs if d == top)          # và là id nhỏ nhất trong các việc đó
print("đáp án luôn là id nhỏ nhất trong các việc dài nhất")

đáp án luôn là id nhỏ nhất trong các việc dài nhất


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cộng dồn thời gian theo từng nhân viên

In [24]:
from typing import List

def sai_cong_don(n: int, logs: List[List[int]]) -> int:
    total = {}
    prev = 0
    for emp, leave in logs:
        total[emp] = total.get(emp, 0) + (leave - prev)       # SAI: cộng dồn
        prev = leave
    top = max(total.values())
    return min(e for e, t in total.items() if t == top)

n, logs = 10, [[0, 3], [2, 5], [0, 9], [1, 15]]
print("đúng:", Solution().hardestWorker(n, logs), "| sai:", sai_cong_don(n, logs))
assert Solution().hardestWorker(n, logs) != sai_cong_don(n, logs)

đúng: 1 | sai: 0


**Vì sao sai:** đề hỏi nhân viên làm **một công việc** dài nhất, không phải nhân viên có **tổng** thời gian lớn nhất. Id `0` có `3 + 4 = 7` nhưng mỗi việc chỉ `3` và `4`, nhỏ hơn việc `6` của id `1`.

### Lỗi 2: Quên điểm bắt đầu `0` của công việc đầu tiên

In [25]:
def sai_bo_viec_dau(n: int, logs: List[List[int]]) -> int:
    best_id, best_dur = -1, 0
    for i in range(1, len(logs)):                              # SAI: bỏ qua việc đầu
        dur = logs[i][1] - logs[i - 1][1]
        if dur > best_dur or (dur == best_dur and logs[i][0] < best_id):
            best_id, best_dur = logs[i][0], dur
    return best_id

n, logs = 2, [[0, 10], [1, 20]]
print("đúng:", Solution().hardestWorker(n, logs), "| sai:", sai_bo_viec_dau(n, logs))
assert Solution().hardestWorker(n, logs) != sai_bo_viec_dau(n, logs)

đúng: 0 | sai: 1


**Vì sao sai:** việc đầu tiên có thời lượng `leaveTime_0 - 0` chứ không phải bị bỏ. Ở ví dụ 3 việc đầu (`10`) hòa với việc thứ hai (`10`) và `id 0` nhỏ hơn, nhưng code sai chỉ xét việc thứ hai nên trả `1`. Hãy khởi tạo `prev = 0`.

### Lỗi 3: Hòa mà chọn `id` lớn hơn (dùng `>=`)

In [26]:
def sai_hoa_chon_lon(n: int, logs: List[List[int]]) -> int:
    prev, best_id, best_dur = 0, -1, 0
    for emp, leave in logs:
        dur = leave - prev
        if dur >= best_dur:                                    # SAI: hòa thì luôn ghi đè
            best_id, best_dur = emp, dur
        prev = leave
    return best_id

n, logs = 2, [[0, 10], [1, 20]]
print("đúng:", Solution().hardestWorker(n, logs), "| sai:", sai_hoa_chon_lon(n, logs))
assert Solution().hardestWorker(n, logs) != sai_hoa_chon_lon(n, logs)

đúng: 0 | sai: 1


**Vì sao sai:** khi hòa, đề yêu cầu `id` **nhỏ nhất**, không phải việc đến sau. Ghi đè bằng `>=` làm kết quả là `id` của việc sau cùng trong số các việc hòa (ở đây `1`). Hãy so sánh `id` khi hòa (hoặc dùng `>` nếu bạn chắc rằng `id` nhỏ luôn đứng trước, điều **không** được đảm bảo).

### Lỗi 4: Dùng thời điểm kết thúc thay cho thời lượng

In [27]:
def sai_dung_leave(n: int, logs: List[List[int]]) -> int:
    best = max(logs, key=lambda x: x[1])                       # SAI: lấy thời điểm kết thúc lớn nhất
    return best[0]

n, logs = 26, [[1, 1], [3, 7], [2, 12], [7, 17]]
print("đúng:", Solution().hardestWorker(n, logs), "| sai:", sai_dung_leave(n, logs))
assert Solution().hardestWorker(n, logs) != sai_dung_leave(n, logs)

đúng: 3 | sai: 7


**Vì sao sai:** thời điểm kết thúc lớn nhất luôn thuộc việc **cuối cùng**; nhưng việc cuối không nhất thiết là việc dài nhất. Ở ví dụ 2, việc cuối do id `7` chỉ dài `5`, còn id `3` dài `6`. Phải trừ `prev` để có thời lượng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Dữ liệu cho dưới dạng **thời điểm** (mốc) ⇒ nghĩ ngay tới **hiệu hai mốc liên tiếp** (thời lượng), cần một biến `prev`.
- "Dài nhất, hòa thì nhỏ nhất" ⇒ phá hòa bằng so sánh có điều kiện `dur == best_dur and emp < best_id` hoặc khóa `(dur, -id)`.
- Chú ý đề nói **công việc** hay **nhân viên**: có cộng dồn hay không. Ở bài này **không** cộng dồn.
- Bắt đầu `prev = 0` rồi dùng cùng một công thức cho mọi việc, tránh xử lý riêng việc đầu.
- Nên thử ví dụ có cùng `id` xuất hiện hai lần (như ví dụ 1) và ví dụ hòa (ví dụ 3).
- Tự kiểm tra ràng buộc: `leaveTime` tăng nghiêm ngặt nên mọi thời lượng `>= 1`, có thể khởi tạo `best_dur = 0`.

## 15. Biến thể của bài

1. **Nhân viên có tổng thời gian làm việc lớn nhất** (cộng dồn): đổi sang dict `total`.
2. **Việc ngắn nhất** thay vì dài nhất.
3. **Có thời gian nghỉ** giữa các việc (`startTime` cho từng việc): thời lượng là `end - start`.
4. **Top-k việc dài nhất** và các nhân viên tương ứng.
5. **Nhiều nhân viên làm song song** (các khoảng thời gian chồng lên nhau): bài toán khoảng, cần sắp xếp theo thời điểm.
6. **Trả về chỉ số công việc** thay vì `id` nhân viên.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai lượt | tính `durations`, tìm max, lấy `id` nhỏ nhất | `O(m)` | `O(m)` | dễ hiểu, dễ debug |
| 2. Một lượt | giữ `prev` và `(best_id, best_dur)` | `O(m)` | `O(1)` | cách chuẩn, tối ưu bộ nhớ |
| 3. Sắp xếp | `sort` theo `(-dur, id)`, lấy đầu | `O(m log m)` | `O(m)` | khi cần cả thứ hạng |
| 4. `max` với khóa | khóa `(dur, -id)` | `O(m)` | `O(m)` | code ngắn |
| 5. Dict theo `id` | thời lượng một-việc lớn nhất mỗi người | `O(m)` | `O(k)` | khi cần thống kê theo nhân viên |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `Solution().hardestWorker(3, [[1, 4], [2, 6]])` là gì?

**Bài 2 (tính tay).** Với `logs = [[2, 2], [0, 8], [1, 10], [0, 16]]`, viết bốn thời lượng và chọn đáp án.

**Bài 3 (điền TODO).** Hoàn thành hàm `thoi_luong(logs)` trả về danh sách thời lượng.

In [28]:
def thoi_luong(logs):
    prev = 0
    result = []
    for emp, leave in logs:
        # TODO: thêm (leave - prev) vào result rồi cập nhật prev
        pass
    return result

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `tong_thoi_gian_lon_nhat(n, logs)` trả về `id` của nhân viên có **tổng** thời gian làm việc lớn nhất (hòa thì `id` nhỏ nhất). So sánh với `hardestWorker` trên ví dụ 1.

**Bài 5 (biến thể / mini challenge).** Viết `top_k_workers(n, logs, k)` trả về danh sách `k` giá trị `id` (có thể trùng nếu một người làm nhiều việc dài) ứng với `k` việc dài nhất, sắp xếp theo `(thời lượng giảm dần, id tăng dần)`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: thời lượng `4` (id 1) và `2` (id 2). Bài 2: `2, 6, 2, 6`.
- **Mức 2 (vừa):** Bài 3: `result.append(leave - prev); prev = leave`. Bài 4: dùng `dict` cộng dồn thay cho `max`.
- **Mức 3 (sâu):** Bài 5: tạo danh sách `(emp, dur)` rồi `sorted(..., key=lambda x: (-x[1], x[0]))[:k]`; nhớ `k` có thể lớn hơn số việc.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1: id 1 làm 4, id 2 làm 2 -> 1
assert Solution().hardestWorker(3, [[1, 4], [2, 6]]) == 1

# Bài 2: 2, 6, 2, 6 -> hòa 6 giữa id 0 (việc 1) và id 0 (việc 3) -> 0
assert Solution().hardestWorker(3, [[2, 2], [0, 8], [1, 10], [0, 16]]) == 0

# Bài 3
def thoi_luong(logs):
    prev = 0
    result = []
    for emp, leave in logs:
        result.append(leave - prev)
        prev = leave
    return result

# Bài 4
def tong_thoi_gian_lon_nhat(n, logs):
    total, prev = {}, 0
    for emp, leave in logs:
        total[emp] = total.get(emp, 0) + (leave - prev)
        prev = leave
    top = max(total.values())
    return min(e for e, t in total.items() if t == top)

# Bài 5
def top_k_workers(n, logs, k):
    prev, tasks = 0, []
    for emp, leave in logs:
        tasks.append((emp, leave - prev))
        prev = leave
    tasks.sort(key=lambda x: (-x[1], x[0]))
    return [e for e, _ in tasks[:k]]

assert thoi_luong([[0, 3], [2, 5], [0, 9], [1, 15]]) == [3, 2, 4, 6]
assert thoi_luong([[0, 7]]) == [7]
assert tong_thoi_gian_lon_nhat(10, [[0, 3], [2, 5], [0, 9], [1, 15]]) == 0      # id 0 tổng 7
assert tong_thoi_gian_lon_nhat(2, [[0, 10], [1, 20]]) == 0
assert top_k_workers(10, [[0, 3], [2, 5], [0, 9], [1, 15]], 2) == [1, 0]        # 6 (id 1), 4 (id 0)
assert top_k_workers(2, [[0, 10], [1, 20]], 5) == [0, 1]
assert top_k_workers(26, [[1, 1], [3, 7], [2, 12], [7, 17]], 3) == [3, 2, 7]    # 6, 5, 5 -> id 2 trước id 7
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (một lượt với `prev` và phá hòa theo `id`)

```python
class Solution:
    def hardestWorker(self, n: int, logs: List[List[int]]) -> int:
        prev = 0
        best_id, best_dur = -1, 0
        for emp, leave in logs:
            dur = leave - prev
            if dur > best_dur or (dur == best_dur and emp < best_id):
                best_id, best_dur = emp, dur
            prev = leave
        return best_id
```

**Checklist**

- [ ] Khởi tạo `prev = 0` để việc đầu tiên có thời lượng `leaveTime_0`?
- [ ] Cập nhật `prev = leave` **ở cuối mỗi vòng lặp** (sau khi so sánh)?
- [ ] Khi hòa thời lượng, chọn `id` **nhỏ nhất** (`emp < best_id`)?
- [ ] Không cộng dồn thời gian theo nhân viên (đề hỏi một công việc)?
- [ ] Đã thử `logs` chỉ có một việc và trường hợp hòa như ví dụ 3?
- [ ] Trả về `id`, không phải thời lượng?
- [ ] Signature đúng `hardestWorker(self, n: int, logs: List[List[int]]) -> int` và có `from typing import List`?